# Pre-Training on SlimPajama-6B (Azure)

**Model**: Decoder-only Transformer with GQA + REPO-Attention + Flash-Attention  
**Dataset**: [SlimPajama-6B](https://hub.oxen.ai/datasets/SlimPajama-6B) via Oxen  
**Tracking**: Weights & Biases  

### Dataset Schema (from Oxen)
| Column | Type | Description |
|--------|------|-------------|
| `text`  | str | Raw document text |
| `meta`  | struct | Contains `redpajama_set_name` (C4, CommonCrawl, StackExchange, etc.) |
| `__index_level_0__` | int | Row index |

Uses existing code from `train/` folder: `tokenizer.py`, `dataset_define.py`, `save_checkpoint.py`, and `transformer/build_transformer.py`.

## 0. Install Dependencies (run once on Azure)

In [1]:
# Uncomment and run on Azure VM if packages are missing
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install oxen wandb transformers tokenizers pyarrow

Looking in indexes: https://download.pytorch.org/whl/cu121


## 1. Setup Paths & Imports

In [2]:
import os
import sys
import time
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from datetime import datetime
import wandb

# ---- Set PROJECT_ROOT to the Transformers folder ----
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
TRAIN_DIR = os.path.join(PROJECT_ROOT, "train")

# Add both to sys.path so we can import our existing modules
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, TRAIN_DIR)

# ---- Import YOUR existing code ----
# transformer/ is a proper package with __init__.py
from transformer.build_transformer import build_transformer

# train/ files are imported directly (TRAIN_DIR is on sys.path)
from dataset_define import SlimPajamaDataset
from save_checkpoint import save_checkpoint
from tokenizer import tokenizer

print(f"Project root  : {PROJECT_ROOT}")
print(f"Train dir     : {TRAIN_DIR}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")

/home/spedrox/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pad token: [PAD], ID: 50257
Added 3 special tokens
Project root  : /home/spedrox/Transformers
Train dir     : /home/spedrox/Transformers/train
PyTorch       : 2.5.1+cu121
CUDA available: True
GPU           : NVIDIA A100 80GB PCIe


## 2. Configuration

In [3]:
# ======================== PATHS ========================
DATASET_DIR    = os.path.join(PROJECT_ROOT, "SlimPajama-6B")  # where oxen clones to
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")

# ======================== MODEL ========================
D_MODEL    = 768
NUM_LAYERS = 12
NUM_HEADS  = 12
KV_HEADS   = 4
D_FF       = 3072
DROPOUT    = 0.1
MAX_SEQ_LEN = 2048

USE_REPO  = True    # REPO-Attention (learned positions)
USE_FLASH = True    # Flash-Attention (PyTorch >= 2.0)

# ======================== TRAINING ========================
EPOCHS          = 3
BATCH_SIZE      = 20    # Lower micro-batch to prevent CUDA OOM at seq_len=2048
GRAD_ACCUM      = 8   # Effective batch size = BATCH_SIZE * GRAD_ACCUM
LEARNING_RATE   = 3e-4
MIN_LR_RATIO    = 0.1  # Final LR = LEARNING_RATE * MIN_LR_RATIO
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 1.0
WARMUP_STEPS    = 2000
ADAM_BETAS      = (0.9, 0.95)
ADAM_EPS        = 1e-8
LABEL_SMOOTHING = 0.0   # No label smoothing for pre-training
ESTIMATED_TOKENS = 6_000_000_000  # SlimPajama-6B total tokens
ESTIMATED_TOTAL_STEPS = ESTIMATED_TOKENS // (BATCH_SIZE * GRAD_ACCUM * MAX_SEQ_LEN)  # ~15k steps/epoch

# ======================== WANDB ========================
WANDB_PROJECT = "Spedrox_llm"
USE_WANDB     = True

# ======================== DEVICE ========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tokenizer info
VOCAB_SIZE = len(tokenizer)
PAD_TOKEN_ID = tokenizer.pad_token_id

print(f"Vocab size    : {VOCAB_SIZE}")
print(f"Pad token ID  : {PAD_TOKEN_ID}")
print(f"Device        : {device}")
print(f"Peak LR       : {LEARNING_RATE}")
print(f"Min LR ratio  : {MIN_LR_RATIO}")
print(f"Dataset dir   : {DATASET_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

Vocab size    : 50260
Pad token ID  : 50257
Device        : cuda
Peak LR       : 0.0002
Min LR ratio  : 0.1
Dataset dir   : /home/spedrox/Transformers/SlimPajama-6B
Checkpoint dir: /home/spedrox/Transformers/checkpoints


## 3. Clone SlimPajama-6B via Oxen

In [4]:
import oxen
from oxen.auth import config_auth

# Replace with your actual API key from Oxen.ai profile
config_auth("SFMyNTY.g2gDbQAAAC9hcGlfa2V5X3YxOjUxZDI0YzRiLTBhODUtNGNjNS1hZDM5LThjNzlkY2Q2NDMwMm4GAOPvxE2dAWIAAVGA.WSKjFQnc7B_De_Wj2IWmkKtzzTQTjqbdfhH91MLJeiM")

DATASET_DIR = "./SlimPajama-6B"

if not os.path.exists(DATASET_DIR):
    print("Cloning SlimPajama-6B dataset via Oxen...")
    repo = oxen.clone("datasets/SlimPajama-6B", path=DATASET_DIR)
    print("Clone complete!")
else:
    print(f"Dataset already exists at {DATASET_DIR}")

# List what we got
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, "").count(os.sep)
    if level < 2:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in files[:10]:
            print(f"{indent}  {f}")
        if len(files) > 10:
            print(f"{indent}  ... and {len(files)-10} more files")

Dataset already exists at ./SlimPajama-6B
SlimPajama-6B/
  README.md
  .oxen/
    config.toml
    HEAD
  data/
    train-00023-of-00048-792da495e1006029.parquet
    test-00000-of-00001-9f769cf7ce219017.parquet
    train-00002-of-00048-4ff3ce9cc290a2f2.parquet
    train-00000-of-00048-ab2b35705f029d94.parquet
    train-00025-of-00048-4795c46808b8c271.parquet
    validation-00000-of-00001-4fb685c22a3f91ef.parquet
    train-00007-of-00048-063b7f39694e5033.parquet
    train-00017-of-00048-ddf85b481214bd2b.parquet
    train-00037-of-00048-004d930af7b6e3b6.parquet
    train-00027-of-00048-34be397c8ed8c605.parquet
    ... and 40 more files


## 4. Build Model (using your `build_transformer`)

In [5]:
model = build_transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    src_seq_len=MAX_SEQ_LEN,
    tgt_seq_len=MAX_SEQ_LEN,
    d_model=D_MODEL,
    N=NUM_LAYERS,
    h=NUM_HEADS,
    kv_h=KV_HEADS,
    dropout=DROPOUT,
    d_ff=D_FF,
    use_repo=USE_REPO,
    use_flash=USE_FLASH,
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters     : {total_params:,} ({total_params/1e6:.1f}M)")
print(f"Trainable parameters : {trainable_params:,}")
print(f"REPO-Attention       : {'ON' if USE_REPO else 'OFF'}")
print(f"Flash-Attention      : {'ON' if USE_FLASH else 'OFF'}")

if torch.cuda.is_available():
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Total parameters     : 154,582,996 (154.6M)
Trainable parameters : 154,582,996
REPO-Attention       : ON
Flash-Attention      : ON
GPU                  : NVIDIA A100 80GB PCIe
GPU Memory           : 85.0 GB


## 5. Sanity Check (forward + backward with dummy data)

In [6]:
print("Running sanity check...")
model.train()

dummy_ids = torch.randint(0, VOCAB_SIZE, (2, MAX_SEQ_LEN), device=device)
dummy_labels = torch.randint(0, VOCAB_SIZE, (2, MAX_SEQ_LEN), device=device)

# Forward (same logic as your train.py)
embeddings = model.tgt_embed(dummy_ids)
output = embeddings
for layer in model.decoder.layers:
    output, _ = layer(output, tgt_mask=None, use_cache=False)
output = model.decoder.norm(output)
logits = model.project(output)

# Loss (same as your train.py)
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = dummy_labels[..., 1:].contiguous()
loss = nn.CrossEntropyLoss(ignore_index=-100)(
    shift_logits.view(-1, shift_logits.size(-1)),
    shift_labels.view(-1)
)

# Backward
loss.backward()

print(f"[PASS] logits shape : {logits.shape}")
print(f"[PASS] loss         : {loss.item():.4f}")
print(f"[PASS] loss finite  : {torch.isfinite(loss).item()}")
print(f"[PASS] grads OK     : {all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters() if p.requires_grad)}")

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Sanity check passed!")

Running sanity check...
[PASS] logits shape : torch.Size([2, 2048, 50260])
[PASS] loss         : 10.8250
[PASS] loss finite  : True
[PASS] grads OK     : True
Sanity check passed!


## 6. Create Dataset & DataLoader (using your `SlimPajamaDataset`)

In [7]:
train_dataset = SlimPajamaDataset(
    data_dir=DATASET_DIR,
    tokenizer=tokenizer,
    max_length=MAX_SEQ_LEN,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=16,      # Tuned for 24 vCores
    pin_memory=True,
    prefetch_factor=2,
)

print(f"DataLoader ready (batch_size={BATCH_SIZE}, num_workers=4)")

Found 50 parquet files in ./SlimPajama-6B
DataLoader ready (batch_size=20, num_workers=4)


## 7. WandB Init

In [8]:
wandb.login(key="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")

if USE_WANDB:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f"slimpajama_pretrain_{timestamp}"
    
    wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            "model_type": "decoder_only_transformer",
            "d_model": D_MODEL,
            "num_layers": NUM_LAYERS,
            "num_heads": NUM_HEADS,
            "num_kv_heads": KV_HEADS,
            "d_ff": D_FF,
            "vocab_size": VOCAB_SIZE,
            "max_sequence_length": MAX_SEQ_LEN,
            "dropout": DROPOUT,
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "weight_decay": WEIGHT_DECAY,
            "gradient_clipping": MAX_GRAD_NORM,
            "warmup_steps": WARMUP_STEPS,
            "dataset": "SlimPajama-6B",
            "dataset_source": "oxen.ai",
            "mixed_precision": True,
            "device": str(device),
            "architecture_features": ["GQA", "REPO-Attention", "Flash-Attention", "RMSNorm"],
            "total_params": total_params,
        },
        tags=["pre-training", "slimpajama", "gqa", "repo-attention", "flash-attention"]
    )
    wandb.watch(model, log="all", log_freq=200)
    print(f"WandB run started: {wandb.run.url}")
else:
    print("WandB disabled.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/spedrox/.netrc
wandb: Currently logged in as: dinmaybrahmaofficial (dinmaybrahmaofficial-indian-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB run started: https://wandb.ai/dinmaybrahmaofficial-indian-institute-of-technology/Spedrox_llm/runs/op4r7g8h


## 8. Training Loop

Uses the same training logic from your `train/train.py`:
- Mixed precision via `autocast` + `GradScaler`
- Same forward pass: `tgt_embed` -> decoder layers -> norm -> project
- Same loss: `CrossEntropyLoss(ignore_index=pad_token_id)`
- Same checkpoint saving via your `save_checkpoint()`
- Auto-checkpoint every 2 hours

In [ ]:
# ======================== TRAINING ========================

import math
from torch.optim.lr_scheduler import LambdaLR

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=ADAM_BETAS,
    eps=ADAM_EPS,
)

# train_loader can be Sized or Iterable; build scheduler accordingly.
try:
    batches_per_epoch = len(train_loader)
    updates_per_epoch = math.ceil(batches_per_epoch / GRAD_ACCUM)
    total_updates = max(1, updates_per_epoch * EPOCHS)
except TypeError:
    # IterableDataset: estimate from known token count
    total_updates = ESTIMATED_TOTAL_STEPS * EPOCHS
    print(f"IterableDataset detected. Estimated total optimizer steps: {total_updates}")

warmup_updates = min(WARMUP_STEPS, total_updates)
has_known_total_updates = True  # Always use cosine decay

def lr_lambda(step: int) -> float:
    step = step + 1  # LambdaLR passes 0-based step index.

    if step <= warmup_updates:
        return float(step) / float(max(1, warmup_updates))

    if has_known_total_updates:
        if total_updates <= warmup_updates:
            return 1.0
        progress = (step - warmup_updates) / max(1, total_updates - warmup_updates)
        progress = min(max(progress, 0.0), 1.0)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine

    # Should not reach here since has_known_total_updates is always True now.
    return MIN_LR_RATIO

scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

# Start from warmup LR immediately so first optimizer step is not done at peak LR.
initial_lr_scale = lr_lambda(0)
for param_group in optimizer.param_groups:
    param_group["lr"] = LEARNING_RATE * initial_lr_scale

autocast_enabled = device.type == "cuda"
autocast_dtype = torch.bfloat16 if (autocast_enabled and torch.cuda.is_bf16_supported()) else torch.float16
use_grad_scaler = autocast_enabled and autocast_dtype == torch.float16
scaler = GradScaler(enabled=use_grad_scaler)

if not autocast_enabled:
    mp_mode = "OFF"
elif autocast_dtype == torch.bfloat16:
    mp_mode = "BF16"
else:
    mp_mode = "FP16"

criterion = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=LABEL_SMOOTHING)

model.train()
global_step = 0
best_loss = float("inf")
avg_loss = float("inf")
last_checkpoint_time = time.time()
epoch_losses = []

optimizer.zero_grad(set_to_none=True)

print("Starting training...")
print(f"  Epochs         : {EPOCHS}")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Grad accum     : {GRAD_ACCUM}")
print(f"  Peak LR        : {LEARNING_RATE}")
print(f"  Warmup steps   : {warmup_updates}")
print(f"  Total updates  : {total_updates if total_updates is not None else 'unknown'}")
print(f"  Mixed precision: {mp_mode}")
print()

for epoch in range(EPOCHS):
    total_loss = 0.0
    epoch_start_time = time.time()
    batch_count = 0
    accum_window_loss = 0.0
    accum_window_batches = 0
    micro_batches_in_accum = 0
    last_grad_norm_value = None

    for i, batch in enumerate(train_loader):
        current_time = time.time()

        # ---- Auto-save every 2 hours ----
        if current_time - last_checkpoint_time >= 7200:
            print(f"\nAuto-saving checkpoint at epoch {epoch + 1}, batch {i}...")
            avg_loss = total_loss / max(batch_count, 1)
            save_checkpoint(
                model, optimizer, epoch, global_step, avg_loss, best_loss,
                CHECKPOINT_DIR, f"auto_checkpoint_epoch_{epoch + 1}_step_{global_step}.pt"
            )
            last_checkpoint_time = current_time

        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        if autocast_enabled:
            amp_ctx = autocast(device_type="cuda", dtype=autocast_dtype)
        else:
            amp_ctx = autocast(device_type=device.type, enabled=False)

        # ---- Forward pass ----
        with amp_ctx:
            embeddings = model.tgt_embed(input_ids)
            output = embeddings
            for layer in model.decoder.layers:
                output, _ = layer(output, tgt_mask=None, use_cache=False)
            output = model.decoder.norm(output)
            logits = model.project(output)

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()

            loss = criterion(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )
            loss = loss / GRAD_ACCUM

        if not torch.isfinite(loss):
            print(f"[WARN] Non-finite loss at epoch {epoch + 1}, batch {i}. Skipping batch.")
            optimizer.zero_grad(set_to_none=True)
            micro_batches_in_accum = 0
            accum_window_loss = 0.0
            accum_window_batches = 0
            continue

        # ---- Backward ----
        if scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()

        unscaled_loss = loss.item() * GRAD_ACCUM
        total_loss += unscaled_loss
        batch_count += 1
        epoch_losses.append(unscaled_loss)
        accum_window_loss += unscaled_loss
        accum_window_batches += 1
        micro_batches_in_accum += 1

        should_step = micro_batches_in_accum >= GRAD_ACCUM

        if should_step:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            grad_norm_value = float(grad_norm.item() if torch.is_tensor(grad_norm) else grad_norm)
            last_grad_norm_value = grad_norm_value

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            micro_batches_in_accum = 0

            window_loss = accum_window_loss / max(1, accum_window_batches)
            accum_window_loss = 0.0
            accum_window_batches = 0

            # ---- WandB logging (every optimizer update) ----
            if USE_WANDB:
                log_dict = {
                    "train/loss": window_loss,
                    "train/epoch": epoch + 1,
                    "train/global_step": global_step,
                    "train/learning_rate": optimizer.param_groups[0]["lr"],
                    "train/grad_norm": grad_norm_value,
                }
                if torch.cuda.is_available():
                    log_dict.update({
                        "system/gpu_memory_allocated_gb": torch.cuda.memory_allocated() / 1e9,
                        "system/gpu_memory_reserved_gb": torch.cuda.memory_reserved() / 1e9,
                    })
                wandb.log(log_dict, step=global_step)

        # ---- Print progress ----
        if i % 5 == 0:
            elapsed_time = time.time() - epoch_start_time
            gpu_memory = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
            lr_now = optimizer.param_groups[0]["lr"]
            grad_norm_msg = (
                f", GradNorm: {last_grad_norm_value:.2f}" if last_grad_norm_value is not None else ""
            )
            print(
                f"Epoch {epoch + 1}, Batch {i}, Loss: {unscaled_loss:.4f}, "
                f"LR: {lr_now:.6f}, Time: {elapsed_time:.1f}s, Step: {global_step}, "
                f"GPU: {gpu_memory:.1f}GB{grad_norm_msg}"
            )

        if i % 10 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Flush leftover gradients at epoch end if batch count is not divisible by GRAD_ACCUM.
    if micro_batches_in_accum > 0:
        if scaler.is_enabled():
            scaler.unscale_(optimizer)

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        grad_norm_value = float(grad_norm.item() if torch.is_tensor(grad_norm) else grad_norm)
        last_grad_norm_value = grad_norm_value

        if scaler.is_enabled():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

        window_loss = accum_window_loss / max(1, accum_window_batches)
        accum_window_loss = 0.0
        accum_window_batches = 0
        micro_batches_in_accum = 0

        if USE_WANDB:
            log_dict = {
                "train/loss": window_loss,
                "train/epoch": epoch + 1,
                "train/global_step": global_step,
                "train/learning_rate": optimizer.param_groups[0]["lr"],
                "train/grad_norm": grad_norm_value,
            }
            if torch.cuda.is_available():
                log_dict.update({
                    "system/gpu_memory_allocated_gb": torch.cuda.memory_allocated() / 1e9,
                    "system/gpu_memory_reserved_gb": torch.cuda.memory_reserved() / 1e9,
                })
            wandb.log(log_dict, step=global_step)

    # ---- End of epoch ----
    avg_loss = total_loss / max(batch_count, 1)
    epoch_duration = time.time() - epoch_start_time
    print(f"Epoch {epoch + 1}/{EPOCHS}, Avg Loss: {avg_loss:.4f}, Duration: {epoch_duration:.1f}s")

    if USE_WANDB:
        wandb.log({
            "epoch/avg_loss": avg_loss,
            "epoch/duration_seconds": epoch_duration,
            "epoch/batches_processed": batch_count,
            "epoch/min_loss": min(epoch_losses[-batch_count:]) if batch_count > 0 else 0,
            "epoch/max_loss": max(epoch_losses[-batch_count:]) if batch_count > 0 else 0,
        }, step=global_step)

    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"New best loss: {best_loss:.4f} - Saving best model...")
        save_checkpoint(
            model, optimizer, epoch, global_step, avg_loss, best_loss,
            CHECKPOINT_DIR, "best_model.pt"
        )
        if USE_WANDB:
            wandb.log({"train/best_loss": best_loss}, step=global_step)

    save_checkpoint(
        model, optimizer, epoch, global_step, avg_loss, best_loss,
        CHECKPOINT_DIR, f"epoch_{epoch + 1}_checkpoint.pt"
    )

# ---- Final save ----
print("Training completed! Saving final checkpoint...")
save_checkpoint(
    model, optimizer, EPOCHS - 1, global_step, avg_loss, best_loss,
    CHECKPOINT_DIR, "final_model.pt"
)

if USE_WANDB:
    wandb.finish()

print("Done!")

train_loader has no __len__ (IterableDataset). Using warmup + inverse-sqrt LR decay.
Starting training...
  Epochs         : 3
  Batch size     : 20
  Grad accum     : 8
  Peak LR        : 0.0002
  Warmup steps   : 1000
  Total updates  : unknown
  Mixed precision: BF16



Token indices sequence length is longer than the specified maximum sequence length for this model (2156 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2371 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1047 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2619 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1225 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence leng

Epoch 1, Batch 0, Loss: 10.8125, LR: 0.000000, Time: 2.8s, Step: 0, GPU: 10.0GB
Epoch 1, Batch 5, Loss: 10.8125, LR: 0.000000, Time: 6.1s, Step: 0, GPU: 10.0GB
Epoch 1, Batch 10, Loss: 10.8125, LR: 0.000000, Time: 8.6s, Step: 1, GPU: 11.3GB, GradNorm: 0.08
Epoch 1, Batch 15, Loss: 10.8125, LR: 0.000001, Time: 11.8s, Step: 2, GPU: 10.6GB, GradNorm: 0.07
Epoch 1, Batch 20, Loss: 10.8125, LR: 0.000001, Time: 14.2s, Step: 2, GPU: 11.3GB, GradNorm: 0.07
Epoch 1, Batch 25, Loss: 10.8125, LR: 0.000001, Time: 17.4s, Step: 3, GPU: 11.3GB, GradNorm: 0.08
Epoch 1, Batch 30, Loss: 10.8125, LR: 0.000001, Time: 19.9s, Step: 3, GPU: 11.3GB, GradNorm: 0.08
Epoch 1, Batch 35, Loss: 10.8125, LR: 0.000001, Time: 23.0s, Step: 4, GPU: 11.3GB, GradNorm: 0.08
Epoch 1, Batch 40, Loss: 10.8125, LR: 0.000001, Time: 25.5s, Step: 5, GPU: 11.3GB, GradNorm: 0.07
Epoch 1, Batch 45, Loss: 10.8125, LR: 0.000001, Time: 28.4s, Step: 5, GPU: 11.3GB, GradNorm: 0.07
Epoch 1, Batch 50, Loss: 10.8125, LR: 0.000001, Time: 30.

## 9. Resume from Checkpoint

In [ ]:
# Uncomment to resume training from a checkpoint
"""
RESUME_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")

if os.path.exists(RESUME_PATH):
    checkpoint = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    global_step = checkpoint['global_step']
    best_loss = checkpoint['best_loss']
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resumed from {RESUME_PATH} at step {global_step}, epoch {start_epoch}")
else:
    print(f"No checkpoint found at {RESUME_PATH}")
"""
print("Resume cell ready (uncomment to use).")